# 18. A player-specific prior

Forecast, Phase 1, step 2c. Notebook 15's true talent blends a skater's last 3 seasons with **1,500 minutes of his position's average**. That pull toward average is what keeps one lucky season from turning a fourth-liner into a star. But notebook 17 showed it's the wrong target for some players: top-5 picks 21 and under beat their projection by +0.022 WAR/60, and pre-NHL scoring (NHLe) predicted the miss even after age and draft.

**The idea:** instead of pulling everyone toward the same average, pull each skater toward **his own expected level**: a **prior** built from

- **NHLe:** his best pre-NHL season, converted to NHL points per game (notebook 16),
- **draft position:** log of his overall pick, plus a flag for undrafted players,
- **age** in the target season,
- **NHL experience:** prior NHL seasons with 200+ minutes (capped at 7).

The prior keeps its fixed weight while his NHL minutes pile up, so for a veteran with 4,000 minutes over 3 seasons it barely matters, and for a 19-year-old with 600 minutes it matters a lot. The part that matters most for young players is the "prospect rule".

**Goalies** have no NHLe (they don't score), so they keep notebook 15's league-average prior.

The code: a small optional hook in `src/forecast.py` (who is pulled toward what) and the prior model in `src/development.py`. Results stay private (`data/projections_2026-27.csv`, new rookie rows in `data/dev_ranges.csv`, gitignored). Nothing on the cards, app or README changes.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, GOALS_PER_WIN
from edge_data import build_bio_table
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, model_table, eligible,
                         candidates, fit_rows, first_seasons, wmean, wsd, wquantile, age_bucket, draft_bucket,
                         TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS, DIAG_INPUTS, VERSIONS,
                         AGE_ORDER, DRAFT_ORDER, QS)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where the projection file and the updated ranges file are saved

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}

history = history_table(pd.read_csv(DATA / "war_by_season_2008_2025.csv"), pd.read_csv(DATA / "goalie_war_2008_2025.csv"))
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")
nhle_factors = pd.read_csv(DATA / "nhle_factors.csv")                # notebook 16 (whole window)
nhle_mult = pd.read_csv(DATA / "nhle_age_multipliers.csv")
dev_ranges = pd.read_csv(DATA / "dev_ranges.csv")                    # notebook 17
luck = dev_ranges[dev_ranges["kind"] == "luck_fit"].set_index("group")[["a", "b"]]

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
print("Train targets:", ", ".join(label(t) for t in TRAIN))
print("Test targets: ", ", ".join(label(t) for t in TEST))

Train targets: 2009-10, 2010-11, 2011-12, 2013-14, 2014-15, 2015-16, 2016-17, 2017-18
Test targets:  2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


In [2]:
# Check: with no priors, the new code gives notebook 15's 2026-27 projection
# (the saved file is rounded to 5 decimals, so "the same" means within 0.00001)
curve_full = aging_curve(history, bios)[1]
old_sk = project(skaters, bios, 2026, SK_WEIGHTS, SK_REG, AGING, curve_full)
old_g = project(goalies, bios, 2026, G_WEIGHTS, G_REG, AGING, curve_full)
saved = pd.read_csv(DATA / "true_talent_2026-27.csv").set_index("playerId")["proj_war_60"]
diff = (pd.concat([old_sk, old_g])["proj_war_60"].reindex(saved.index) - saved).abs().max()
assert diff < 1e-5, diff
print(f"No-prior projection matches true_talent_2026-27.csv (largest difference {diff:.1e}, from rounding)")

No-prior projection matches true_talent_2026-27.csv (largest difference 5.0e-06, from rounding)


## 2. NHLe without peeking at the test seasons

The backtest fits everything on target seasons through 2017-18 and tests on 2018-19 onward. That includes NHLe: notebook 16's factors used data through 2025-26, so here they're **refit with data through 2017-18 only** (same method: next-season factors, forward/defense factors, age multipliers). The full-window factors are used only for the final 2026-27 projection.

In [3]:
f_train, m_train = training_nhle(careers, TRAIN_END)
main = ["KHL", "SHL", "AHL", "NCAA", "OHL", "WHL", "QMJHL", "USHL", "Liiga", "NL"]
chosen = nhle_factors[nhle_factors["chosen"]]
cmp = pd.DataFrame({
    "through 2017-18": f_train[f_train["group"] == "all"].set_index("league")["factor"],
    "F, through 2017-18": f_train[f_train["group"] == "F"].set_index("league")["factor"],
    "D, through 2017-18": f_train[f_train["group"] == "D"].set_index("league")["factor"],
    "whole window (nb 16)": chosen[chosen["group"] == "all"].set_index("league")["factor"],
}).reindex(main)
print(cmp.round(3).to_string())
print("\nAge multipliers:")
print(pd.DataFrame({"through 2017-18": m_train.set_index("age_bucket")["multiplier"],
                    "whole window (nb 16)": nhle_mult.set_index("age_bucket")["multiplier"]}).round(3).T.to_string())

nhle_train = best_nhle(careers, f_train, m_train)
nhle_full = best_nhle(careers, nhle_factors, nhle_mult)

        through 2017-18  F, through 2017-18  D, through 2017-18  whole window (nb 16)
league                                                                               
KHL               0.749               0.745                 NaN                 0.687
SHL               0.594               0.536                 NaN                 0.552
AHL               0.470               0.453               0.520                 0.466
NCAA              0.393               0.374               0.460                 0.398
OHL               0.304               0.301                 NaN                 0.300
WHL               0.312               0.306               0.313                 0.290
QMJHL             0.180                 NaN                 NaN                 0.265
USHL              0.322                 NaN                 NaN                 0.323
Liiga             0.484                 NaN                 NaN                 0.431
NL                0.485                 NaN           

## 3. The prior model

For each position, a minutes-weighted regression of **actual GAR/60** on the five inputs, using every skater-season with 300+ minutes in the training target seasons (rookies included). Players with no pre-NHL season of 20+ games get their position's average NHLe; undrafted players get the undrafted flag.

**Why log of the pick:** going from pick 1 to pick 5 is a big step, from pick 150 to 200 a tiny one. The log does exactly that with one number, which matches notebook 17 (the draft effect sits at the very top).

**Expected signs:** higher NHLe, an earlier pick (smaller log), more experience: higher prior. Undrafted: lower. **Age:** at the same experience, a player who reaches the NHL older is usually a later bloomer or a depth signing, so age should be negative.

In [4]:
# Features for every skater the projection covers, plus every skater-season used for fitting
pairs = pd.concat([candidates(history, TARGETS + [2026]),
                   skaters[skaters["season"].isin(TARGETS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat_train = skater_features(pairs, history, bios, profiles, nhle_train)
feat_full = skater_features(pairs, history, bios, profiles, nhle_full)
print(f"Pre-NHL NHLe found for {feat_train['nhle_ppg'].notna().mean():.1%} of skater-seasons")

train_rows = fit_rows(history, feat_train, TRAIN)
models = {v: fit_prior(train_rows[eligible(train_rows, v)], DIAG_INPUTS if v == "d" else PRIOR_INPUTS) for v in VERSIONS}
print(f"Fit on {len(train_rows):,} skater-seasons (version a). Weighted R^2: "
      + ", ".join(f"{NAMES[g]} {m['r2']:.3f}" for g, m in models["a"].items()))
model_table(models["a"]).round(4)

Pre-NHL NHLe found for 99.7% of skater-seasons
Fit on 5,152 skater-seasons (version a). Weighted R^2: Defensemen 0.078, Forwards 0.150


D               F        
              coef      se    coef      se
constant    0.3182  0.0385  0.6877  0.0467
nhle_ppg    0.3477  0.0431  0.6053  0.0443
log_pick    0.0145  0.0039 -0.0247  0.0045
undrafted   0.0747  0.0211 -0.0801  0.0279
age        -0.0174  0.0018 -0.0284  0.0022
experience  0.0189  0.0032  0.0326  0.0040

In [5]:
EXPECT = {"nhle_ppg": +1, "log_pick": -1, "undrafted": -1, "age": -1, "experience": +1}
rows = []
for v in ["a", "b"]:
    for g, m in models[v].items():
        for k, sign in EXPECT.items():
            c, s = m["coef"][k], m["se"][k]
            ok = np.sign(c) == sign
            rows.append({"version": v, "group": g, "input": k, "coef": c, "se": s, "z": c / s,
                         "sign": "as expected" if ok else ("WRONG (beyond noise)" if abs(c / s) > 2 else "wrong, within noise")})
signs = pd.DataFrame(rows)
print("Sign check (version a = everyone, b = 3 or fewer NHL seasons):")
signs.round(4)

Sign check (version a = everyone, b = 3 or fewer NHL seasons):


,version,group,input,coef,se,z,sign
0,a,D,nhle_ppg,0.3477,0.0431,8.0738,as expected
1,a,D,log_pick,0.0145,0.0039,3.7584,WRONG (beyond noise)
2,a,D,undrafted,0.0747,0.0211,3.5357,WRONG (beyond noise)
3,a,D,age,-0.0174,0.0018,-9.5126,as expected
4,a,D,experience,0.0189,0.0032,5.8967,as expected
5,a,F,nhle_ppg,0.6053,0.0443,13.6654,as expected
6,a,F,log_pick,-0.0247,0.0045,-5.4302,as expected
7,a,F,undrafted,-0.0801,0.0279,-2.8651,as expected
8,a,F,age,-0.0284,0.0022,-12.7466,as expected
9,a,F,experience,0.0326,0.0040,8.1877,as expected


## 4. Backtest: who should get it, and how much weight?

Four versions, each with the prior at 750, 1,500 or 3,000 minutes of weight (everyone else keeps 1,500 minutes of the position average):

| Version | Gets the prior |
|---|---|
| a | every skater |
| b | skaters with 3 or fewer prior NHL seasons |
| c | skaters 25 or under |
| d | every skater, but the prior uses only age and experience (no NHLe, no draft): a diagnostic |

Version (d) answers a specific question: if (a) helps veterans, is that because of NHLe and draft, or just because the prior knows their age? Error = minutes-weighted RMSE of GAR/60, as in notebook 15, on the same players notebook 15 tested.

**Picking rule:** the lowest error for all skaters among versions (a)-(c), as long as veterans (4+ NHL seasons) don't get worse. **Settings are picked on the test seasons** (2018-19 on), so the winner's test error is a little optimistic. The training-period grid (where the prior is fit on the same seasons it's scored on) is shown next to it to see whether the ranking holds.

In [6]:
cand_feat = lambda tg, feat: candidates(history, tg).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
cand_test, cand_train = cand_feat(TEST, feat_train), cand_feat(TRAIN, feat_train)

def prior_rows(cand, v, model):
    el = cand[eligible(cand, v)]
    return el[["playerId", "season"]].assign(prior_gar_60=apply_prior(el, model[v]).to_numpy())

info = feat_train[["playerId", "season", "age", "experience"]]
def run(targets, priors=None, reg=None):
    b = backtest(skaters, targets, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=priors, prior_reg=reg)
    return b.merge(info, on=["playerId", "season"], how="left")

def score(b):
    rm = lambda m: np.sqrt(np.average((b["actual"][m] - b["pred"][m]) ** 2, weights=b["toi"][m]))
    return {"all": rm(b["age"] > 0), "23 and under": rm(b["age"] <= 23), "3 or fewer seasons": rm(b["experience"] <= 3),
            "veterans (4+)": rm(b["experience"] >= 4),
            "bias (pred - actual, WAR/60)": np.average(b["pred"] - b["actual"], weights=b["toi"]) / GOALS_PER_WIN}

grid, tests = [], {}
for period, tg, cand in [("test", TEST, cand_test), ("train", TRAIN, cand_train)]:
    base = run(tg)
    if period == "test":
        tests[("nb15", None)] = base
    grid.append({"period": period, "version": "notebook 15", "weight": 1500, **score(base)})
    for v in VERSIONS:
        pr = prior_rows(cand, v, models)
        for w in [750, 1500, 3000]:
            b = run(tg, pr, w)
            if period == "test":
                tests[(v, w)] = b
            grid.append({"period": period, "version": v, "weight": w, **score(b)})
grid = pd.DataFrame(grid)
print("Test seasons (2018-19 on):")
print(grid[grid["period"] == "test"].drop(columns="period").round(4).to_string(index=False))

Test seasons (2018-19 on):
    version  weight    all  23 and under  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)
notebook 15    1500 0.2732        0.2938              0.2912         0.2649                        0.0022
          a     750 0.2690        0.2850              0.2832         0.2626                        0.0022
          a    1500 0.2670        0.2834              0.2799         0.2612                        0.0013
          a    3000 0.2704        0.2866              0.2818         0.2652                        0.0000
          b     750 0.2705        0.2847              0.2828         0.2649                        0.0020
          b    1500 0.2694        0.2820              0.2793         0.2649                        0.0020
          b    3000 0.2699        0.2834              0.2808         0.2649                        0.0019
          c     750 0.2704        0.2845              0.2847         0.2639                        0.0031
          c    1500

In [7]:
print("Training seasons (prior fit on these same seasons, so in-sample):")
tr = grid[grid["period"] == "train"].drop(columns="period")
tr = tr.assign(rank=tr["all"].rank().astype(int))
print(tr.round(4).to_string(index=False))

t = grid[grid["period"] == "test"].set_index(["version", "weight"])
base_vets = t.loc[("notebook 15", 1500), "veterans (4+)"]
ok = t[t.index.get_level_values(0).isin(["a", "b", "c"]) & (t["veterans (4+)"] <= base_vets + 1e-12)]
V, W = ok["all"].idxmin()
print(f"\nChosen (lowest all-skater test error, veterans not worse): version {V} ({VERSIONS[V]}), {W:,} minutes")
tri = tr.set_index(["version", "weight"])
tv, tw = tri[tri.index.get_level_values(0).isin(["a", "b", "c"])]["all"].idxmin()
print(f"Best on the training seasons (in-sample): version {tv}, {tw:,} minutes. "
      f"Same version: {'yes' if tv == V else 'no'}; same weight: {'yes' if tw == W else 'no'}")

Training seasons (prior fit on these same seasons, so in-sample):
    version  weight    all  23 and under  3 or fewer seasons  veterans (4+)  bias (pred - actual, WAR/60)  rank
notebook 15    1500 0.2587        0.2852              0.2754         0.2493                        0.0063    12
          a     750 0.2571        0.2814              0.2702         0.2498                        0.0066    10
          a    1500 0.2515        0.2745              0.2628         0.2452                        0.0056     2
          a    3000 0.2507        0.2735              0.2614         0.2447                        0.0043     1
          b     750 0.2567        0.2810              0.2700         0.2493                        0.0062     9
          b    1500 0.2540        0.2739              0.2626         0.2493                        0.0059     4
          b    3000 0.2535        0.2726              0.2610         0.2493                        0.0056     3
          c     750 0.2584        0.28

In [8]:
nb15, a_, d_ = t.loc[("notebook 15", 1500)], t.loc[("a", W)], t.loc[("d", W)]
gain_a, gain_d = nb15["veterans (4+)"] - a_["veterans (4+)"], nb15["veterans (4+)"] - d_["veterans (4+)"]
print(f"Veterans: notebook 15 {nb15['veterans (4+)']:.4f}, (d) age + experience only {d_['veterans (4+)']:.4f}, (a) full prior {a_['veterans (4+)']:.4f}")
share = gain_d / gain_a if gain_a > 0 else np.nan
print(f"Version (d) gets {share:.0%} of version (a)'s veteran gain.")
if share >= 0.8:
    print("Verdict: the veteran gain is an aging fix. The prior's age term adds late-career decline that the centered aging curve under-adjusts; NHLe and draft add little for veterans.")
elif share >= 0.4:
    print("Verdict: most of the veteran gain is an aging fix (the centered aging curve under-adjusts late-career decline); NHLe and draft add the rest.")
else:
    print("Verdict: the veteran gain comes mostly from NHLe and draft, not from age.")

Veterans: notebook 15 0.2649, (d) age + experience only 0.2626, (a) full prior 0.2612
Version (d) gets 62% of version (a)'s veteran gain.
Verdict: most of the veteran gain is an aging fix (the centered aging curve under-adjusts late-career decline); NHLe and draft add the rest.


## 5. Built-in check: do notebook 17's biases shrink?

Notebook 17 found young players and top picks beating their projection. Same buckets, same test misses (actual minus projected WAR/60, centered within each position), before (notebook 15) and after (the chosen prior). The young-player and top-pick averages should move toward 0.

In [9]:
def misses(b):
    b = b.merge(bios[["playerId", "draft_overall"]], on="playerId", how="left")
    b["miss"] = (b["actual"] - b["pred"]) / GOALS_PER_WIN
    b["miss_c"] = b["miss"] - b.groupby("group")["miss"].transform(lambda x: np.average(x, weights=b.loc[x.index, "toi"]))
    b["age_bucket"] = age_bucket(b["age"]).to_numpy()
    b["draft_bucket"] = draft_bucket(b["draft_overall"]).to_numpy()
    return b

before, after = misses(tests[("nb15", None)]), misses(tests[(V, W)])
def bucket_means(col, order, sel=None):
    out = {}
    for name, b in [("before", before), ("after", after)]:
        d = b if sel is None else b[sel(b)]
        out[name] = d.groupby(col).apply(lambda x: wmean(x["miss_c"], x["toi"]), include_groups=False)
    n = (before if sel is None else before[sel(before)]).groupby(col).size()
    return pd.DataFrame({"n": n, **out}).reindex(order).dropna(how="all")

print("By age (skaters, centered miss in WAR/60):")
print(bucket_means("age_bucket", AGE_ORDER).round(4).to_string())
print("\nBy draft position:")
print(bucket_means("draft_bucket", DRAFT_ORDER).round(4).to_string())
before["age_x_draft"] = before["age_bucket"] + " | " + before["draft_bucket"]
after["age_x_draft"] = after["age_bucket"] + " | " + after["draft_bucket"]
young_order = [f"{a} | {d}" for a in ["<=21", "22-23"] for d in DRAFT_ORDER]
print("\nSkaters 23 and under, age x draft:")
print(bucket_means("age_x_draft", young_order, lambda b: b["age"] <= 23).round(4).to_string())

By age (skaters, centered miss in WAR/60):
              n  before   after
age_bucket                     
<=21        176  0.0058 -0.0066
22-23       467  0.0010 -0.0025
24-26       991  0.0021  0.0014
27-29       962 -0.0016 -0.0014
30-32       705 -0.0001  0.0019
33+         517 -0.0035  0.0017

By draft position:
                 n  before   after
draft_bucket                      
1-5            423  0.0091  0.0026
6-15           580  0.0038  0.0004
16-32          668  0.0002 -0.0009
33-64          654 -0.0028 -0.0021
65+           1045 -0.0020  0.0014
undrafted      448 -0.0092 -0.0025

Skaters 23 and under, age x draft:
                       n  before   after
age_x_draft                             
<=21 | 1-5          58.0  0.0144 -0.0010
<=21 | 6-15         52.0 -0.0054 -0.0178
<=21 | 16-32        38.0  0.0017 -0.0068
<=21 | 33-64        20.0  0.0095 -0.0012
<=21 | 65+           8.0  0.0129  0.0076
22-23 | 1-5         58.0  0.0118  0.0035
22-23 | 6-15        96.0  0.0045 -0.0

## 6. True rookies: the prior alone

A player in his first NHL season has no NHL history, so his projection **is** the prior. Test: every skater whose first season in our NHL data is 2018-19 or later, with 300+ minutes. Compare the prior (fit through 2017-18) with simply calling him his position's average. Both the chosen version's prior and version (b)'s (fit only on players with 3 or fewer seasons) are shown.

Then the **spread** of rookie misses in notebook 17's scaled units (z = miss / its expected size at his minutes, from notebook 17's luck fit). A z SD above 1 means rookies need a wider range than veterans with the same minutes.

In [10]:
first = first_seasons(history)
rk = skaters[(skaters["season"] == skaters["playerId"].map(first)) & skaters["season"].isin(TEST) & (skaters["toi"] >= 300)]
rk = rk.assign(gar_60=rk["gar"] / rk["toi"] * 60).merge(feat_train.drop(columns="group"), on=["playerId", "season"], how="left")
rk = rk.merge(nhle_train[["playerId", "nhle_league"]], on="playerId", how="left")
rk["position average"] = [group_average(history, s - 1)[g] for s, g in zip(rk["season"], rk["group"])]
rk[f"prior ({V})"] = apply_prior(rk, models[V])
rk["prior (b)"] = apply_prior(rk, models["b"])
print(f"True rookies (first NHL season 2018-19 on, 300+ minutes): {len(rk)} "
      f"({(rk['group'] == 'F').sum()} F, {(rk['group'] == 'D').sum()} D)")
rows = []
for c in dict.fromkeys(["position average", f"prior ({V})", "prior (b)"]):
    for g in ["all", "F", "D"]:
        d = rk if g == "all" else rk[rk["group"] == g]
        e = d[c] - d["gar_60"]
        rows.append({"projection": c, "group": g, "rookies": len(d),
                     "RMSE (GAR/60)": np.sqrt(np.average(e ** 2, weights=d["toi"])),
                     "bias (pred - actual, WAR/60)": np.average(e, weights=d["toi"]) / GOALS_PER_WIN})
pd.DataFrame(rows).set_index(["projection", "group"]).round(4)

True rookies (first NHL season 2018-19 on, 300+ minutes): 207 (121 F, 86 D)


rookies  RMSE (GAR/60)  bias (pred - actual, WAR/60)
projection       group                                                      
position average all        207         0.2965                        0.0041
                 F          121         0.3481                        0.0076
                 D           86         0.2091                       -0.0004
prior (a)        all        207         0.2925                        0.0071
                 F          121         0.3342                        0.0106
                 D           86         0.2253                        0.0026
prior (b)        all        207         0.2899                        0.0061
                 F          121         0.3286                        0.0080
                 D           86         0.2286                        0.0035

In [11]:
# The spread of rookie misses (chosen prior), in notebook 17's scaled units
rk["miss"] = (rk["gar_60"] - rk[f"prior ({V})"]) / GOALS_PER_WIN
rk["z"] = rk["miss"] / np.sqrt(rk["group"].map(luck["a"]) + rk["group"].map(luck["b"]) / rk["toi"])
rookie_rows = []
for g, d in rk.groupby("group"):
    w = d["toi"].to_numpy()
    zc = d["z"] - wmean(d["z"], w)
    row = {"kind": "rookie", "group": g, "bucket": "first NHL season", "n": len(d), "mean_raw": wmean(d["miss"], w),
           "sd": wsd(d["miss"].to_numpy(), w), "z_mean_raw": wmean(d["z"], w), "z_sd": wsd(zc.to_numpy(), w)}
    row.update({f"z_p{int(q * 100)}": v for q, v in zip(QS, wquantile(zc, w, QS))})
    rookie_rows.append(row)
rookie_rows = pd.DataFrame(rookie_rows)
print("Rookie misses vs. the prior, scaled by notebook 17's luck fit (z SD 1.00 = same spread as everyone else):")
rookie_rows.drop(columns=["kind", "bucket"]).round(3)

Rookie misses vs. the prior, scaled by notebook 17's luck fit (z SD 1.00 = same spread as everyone else):


,group,n,mean_raw,sd,z_mean_raw,z_sd,z_p5,z_p10,z_p25,z_p50,z_p75,z_p90,z_p95
0,D,86,-0.003,0.037,-0.051,1.117,-1.764,-1.189,-0.742,-0.122,0.566,1.311,2.056
1,F,121,-0.011,0.055,-0.179,0.947,-1.574,-1.091,-0.591,-0.090,0.499,1.243,1.853


In [12]:
by_lg = rk.groupby("nhle_league").apply(lambda d: pd.Series({
    "rookies": len(d), "bias (pred - actual, WAR/60)": np.average(d[f"prior ({V})"] - d["gar_60"], weights=d["toi"]) / GOALS_PER_WIN}),
    include_groups=False)
print("Bias by the league of each rookie's best pre-NHL season (5+ rookies; positive = over-predicted):")
by_lg[by_lg["rookies"] >= 5].sort_values("rookies", ascending=False).round(4)

Bias by the league of each rookie's best pre-NHL season (5+ rookies; positive = over-predicted):


,rookies,"bias (pred - actual, WAR/60)"
nhle_league,,
WHL,38.0,0.0244
NCAA,35.0,0.0142
OHL,33.0,0.0109
KHL,21.0,-0.0184
SHL,21.0,-0.0016
AHL,17.0,-0.0011
Liiga,11.0,0.0049
USHL,11.0,0.0019
QMJHL,6.0,-0.0015


## 7. The 2026-27 projection

The final prior is refit on **all** target seasons, with notebook 16's whole-window NHLe. Then the 2026-27 projection is rerun with it.

**League-level check** (as in notebook 15, section 7): the minutes-weighted average projection for each position, with and without the prior, weighted by 2025-26 minutes. If a position moves by more than 0.002 WAR/60, the prior is shifted by a constant so the league level matches the old projection (otherwise a simulated league would start higher or lower than the real one).

In [13]:
final_rows = fit_rows(history, feat_full, TARGETS)
final = fit_prior(final_rows[eligible(final_rows, V)], DIAG_INPUTS if V == "d" else PRIOR_INPUTS)
print(f"Final prior (version {V}), fit on {len(final_rows[eligible(final_rows, V)]):,} skater-seasons:")
print(model_table(final).round(4).to_string())

cand26 = cand_feat([2026], feat_full)
pr26 = prior_rows(cand26, V, {V: final})
new_sk = project(skaters, bios, 2026, SK_WEIGHTS, SK_REG, AGING, curve_full, priors=pr26, prior_reg=W)

mins = history[history["season"] == 2025].groupby("playerId")["toi"].sum()
def level(proj):
    p = proj[proj["played_last_season"]]
    return p.groupby("group").apply(lambda d: np.average(d["proj_war_60"], weights=mins.reindex(d.index)), include_groups=False)

lv = pd.DataFrame({"without prior": level(old_sk), "with prior": level(new_sk)})
lv["change"] = lv["with prior"] - lv["without prior"]
print("\nLeague average 2026-27 projection (WAR/60, weighted by 2025-26 minutes):")
print(lv.round(4).to_string())

shift = {}
for g in ["F", "D"]:
    if abs(lv.loc[g, "change"]) > 0.002:
        # The projection moves in a straight line with the prior, so one test nudge gives the exact shift needed
        nudge = pr26.assign(prior_gar_60=pr26["prior_gar_60"] + 0.1)
        moved = level(project(skaters, bios, 2026, SK_WEIGHTS, SK_REG, AGING, curve_full, priors=nudge, prior_reg=W))
        response = (moved[g] - lv.loc[g, "with prior"]) / 0.1      # WAR/60 of league level per 1 GAR/60 of prior
        shift[g] = -lv.loc[g, "change"] / response
if shift:
    grp = cand26.set_index("playerId")["group"]
    pr26["prior_gar_60"] += pr26["playerId"].map(grp).map(shift).fillna(0).to_numpy()
    new_sk = project(skaters, bios, 2026, SK_WEIGHTS, SK_REG, AGING, curve_full, priors=pr26, prior_reg=W)
    lv["with prior, centered"] = level(new_sk)
    print("\nCentered: prior shifted by " + ", ".join(f"{NAMES[g]} {s:+.4f} GAR/60 ({s / GOALS_PER_WIN:+.4f} WAR/60)" for g, s in shift.items()))
    print(lv.round(4).to_string())
    assert all(abs(lv.loc[g, "with prior, centered"] - lv.loc[g, "without prior"]) < 1e-6 for g in shift)
else:
    print("\nNo position moved by more than 0.002 WAR/60, so the prior is not shifted.")

Final prior (version a), fit on 9,237 skater-seasons:
                 D               F        
              coef      se    coef      se
constant    0.2472  0.0308  0.6534  0.0393
nhle_ppg    0.4434  0.0321  0.6624  0.0372
log_pick    0.0036  0.0030 -0.0308  0.0035
undrafted   0.0212  0.0163 -0.1204  0.0214
age        -0.0140  0.0014 -0.0274  0.0018
experience  0.0159  0.0024  0.0383  0.0031

League average 2026-27 projection (WAR/60, weighted by 2025-26 minutes):
       without prior  with prior  change
group                                   
D             0.0213      0.0186 -0.0027
F             0.0520      0.0512 -0.0008

Centered: prior shifted by Defensemen +0.0508 GAR/60 (+0.0085 WAR/60)
       without prior  with prior  change  with prior, centered
group                                                         
D             0.0213      0.0186 -0.0027                0.0213
F             0.0520      0.0512 -0.0008                0.0512


## 8. Hockey checks

Who moves the most, up and down (skaters who played in 2025-26), and the young players everyone asks about: old vs. new projection in WAR/60, with the prior itself and the inputs behind it.

In [14]:
proj = old_sk[["name", "team", "position", "group", "age_next", "played_last_season", "proj_war_60"]].rename(
    columns={"proj_war_60": "old"})
proj["new"] = new_sk["proj_war_60"].reindex(proj.index)
proj["prior"] = new_sk["prior_gar_60"].reindex(proj.index) / GOALS_PER_WIN
proj["prior_applied"] = new_sk["prior_applied"].reindex(proj.index).astype(bool)
proj["change"] = proj["new"] - proj["old"]
proj = proj.join(cand26.set_index("playerId")[["draft_overall", "experience", "nhle_ppg"]])
proj["minutes 2025-26"] = mins.reindex(proj.index)
SHOW = ["name", "team", "position", "age_next", "draft_overall", "experience", "nhle_ppg", "minutes 2025-26", "old", "prior", "new", "change"]
played = proj[proj["played_last_season"]]
print("Biggest moves up:")
print(played.nlargest(15, "change")[SHOW].round(3).to_string(index=False))
print("\nBiggest moves down:")
print(played.nsmallest(15, "change")[SHOW].round(3).to_string(index=False))

Biggest moves up:
             name team position  age_next  draft_overall  experience  nhle_ppg  minutes 2025-26   old  prior   new  change
     Michael Misa  SJS        C      19.0            2.0           1     0.690          576.483 0.033  0.101 0.074   0.040
     James Hagens  BOS        C      20.0            7.0           0     0.655           29.100 0.044  0.080 0.078   0.034
  Igor Chernyshov  SJS        L      21.0           33.0           1     0.753          421.617 0.073  0.085 0.104   0.031
      Cole Hutson  WSH        D      20.0           43.0           1     0.662          244.400 0.034  0.057 0.063   0.029
      Ilya Protas  WSH        L      20.0           75.0           0     0.680           63.567 0.047  0.071 0.071   0.024
   Porter Martone  PHI        R      20.0            6.0           0     0.575          153.367 0.054  0.072 0.079   0.024
   Gabe Perreault  NYR        R      21.0           23.0           1     0.709          783.333 0.050  0.082 0.073   0.02

In [15]:
WATCH = ["Macklin Celebrini", "Matthew Schaefer", "Ivan Demidov", "Matvei Michkov", "Lane Hutson", "Nate Danielson", "Emmitt Finnie"]
print("Players to watch:")
print(proj[proj["name"].isin(WATCH)][SHOW].round(3).to_string(index=False))
missing = sorted(set(WATCH) - set(proj["name"]))
if missing:
    print("Not in the projection (no NHL season in 2023-24 to 2025-26):", ", ".join(missing))
det = played[(played["team"] == "DET") & (played["age_next"] <= 25)].sort_values("change", ascending=False)
print("\nRed Wings 25 and under (played in 2025-26):")
print(det[SHOW].round(3).to_string(index=False))

Players to watch:
             name team position  age_next  draft_overall  experience  nhle_ppg  minutes 2025-26   old  prior   new  change
      Lane Hutson  MTL        D      22.0           62.0           2     0.662         1949.283 0.063  0.055 0.072   0.009
   Nate Danielson  DET        C      22.0            9.0           1     0.374          309.483 0.031  0.045 0.030  -0.000
   Matvei Michkov  PHI        R      22.0            7.0           2     0.603         1202.067 0.070  0.078 0.082   0.012
    Emmitt Finnie  DET        C      21.0          201.0           1     0.461         1271.000 0.043  0.043 0.042  -0.001
Macklin Celebrini  SJS        C      20.0            1.0           2     0.696         1747.900 0.117  0.107 0.138   0.020
     Ivan Demidov  MTL        R      21.0            5.0           1     0.532         1270.800 0.071  0.070 0.085   0.013
 Matthew Schaefer  NYI        D      19.0            1.0           1     0.106         2023.983 0.064  0.016 0.061  -0.00

## 9. Save

- `projections_2026-27.csv`: every player in notebook 15's projection, with the old projection, the prior he's pulled toward (the league average for goalies), the new projection and whether the player-specific prior was applied. `true_talent_2026-27.csv` stays as it is.
- `dev_ranges.csv`: the rookie spread from section 6 is added as rows with `kind = "rookie"`. Earlier rookie rows (from a previous run) are replaced; every other row is checked to be unchanged.

In [16]:
tt_old = pd.read_csv(DATA / "true_talent_2026-27.csv")
g_rows = old_g[["name", "team", "position", "age_next", "played_last_season", "proj_war_60", "prior_gar_60"]].assign(
    new=old_g["proj_war_60"], prior_applied=False)
g_rows = g_rows.rename(columns={"proj_war_60": "old"}).assign(prior=g_rows["prior_gar_60"] / GOALS_PER_WIN)
out = pd.concat([proj, g_rows])[["name", "team", "position", "age_next", "old", "prior", "new", "prior_applied", "played_last_season"]]
out = out.rename(columns={"age_next": "age", "old": "old_proj_war_60", "prior": "prior_war_60", "new": "new_proj_war_60",
                          "played_last_season": "played_2025_26"}).rename_axis("playerId").reset_index()
assert len(out) == len(tt_old) and set(out["playerId"]) == set(tt_old["playerId"])
out.to_csv(OUT / "projections_2026-27.csv", index=False)
print(f"Saved projections_2026-27.csv: {len(out):,} players, prior applied to {out['prior_applied'].sum():,}")

keep = dev_ranges[dev_ranges["kind"] != "rookie"]
ranges = pd.concat([keep, rookie_rows], ignore_index=True)
assert ranges[ranges["kind"] != "rookie"].reset_index(drop=True).equals(keep.reset_index(drop=True))
ranges.to_csv(OUT / "dev_ranges.csv", index=False)
print(f"Saved dev_ranges.csv: {len(keep)} existing rows unchanged + {len(rookie_rows)} rookie rows")

Saved projections_2026-27.csv: 1,328 players, prior applied to 1,196
Saved dev_ranges.csv: 75 existing rows unchanged + 2 rookie rows


## Takeaways
- **The player-specific prior lowers the error for every group.** On the 2018-19 onward test seasons (prior and NHLe fit only on earlier years), giving every skater his own prior at 1,500 minutes of weight cuts the error from 0.2732 to 0.2670 GAR/60 (2.3%):
  - skaters 23 and under: 0.2938 to 0.2834 (3.5%)
  - 3 or fewer NHL seasons: 0.2912 to 0.2799 (3.9%)
  - veterans: 0.2649 to 0.2612 (1.4%)
  - The settings were picked on the test seasons. On the training seasons version (a) also wins, but at 3,000 minutes (in-sample).
  - Limiting the prior to players with 3 or fewer seasons is slightly better for the young (0.2820 and 0.2793) but leaves veterans unchanged.
- **Most of the veteran gain is an aging fix.** A prior built from age and experience alone (version d) gets 62% of the veteran improvement. The real cause is the aging curve under-adjusting late-career decline (notebook 17's survivor bias). The age term in the prior patches it; NHLe and draft add the rest.
- **The signs mostly make hockey sense.** Higher NHLe, an earlier pick (forwards), more experience: higher prior. Older at the same experience: lower.
  - **Exception: defensemen's draft terms.** In the training fit, earlier picks and drafted defensemen get a slightly *lower* prior, beyond noise. The likely cause is survivors: a late-pick or undrafted defenseman who's still in the NHL earned it.
  - In the final whole-window fit, both draft terms for defensemen are within noise.
  - NHLe carries the defense prior.
- **Notebook 17's biases shrink**, on the test misses (centered, WAR/60):
  - Top-5 picks: +0.009 to +0.003.
  - Undrafted: -0.009 to -0.003.
  - Top-5 picks 21 and under: +0.014 to -0.001; at 22-23: +0.012 to +0.004.
  - It overshoots a little for players 21 and under overall (+0.006 to -0.007), and for picks 6-15 aged 21 and under (-0.005 to -0.018, 52 player-seasons).
- **True rookies:** for 207 first NHL seasons (2018-19 on, 300+ minutes), the prior alone beats the position average overall (0.2925 vs. 0.2965 GAR/60; version (b)'s prior 0.2899).
  - It helps forwards (0.334 vs. 0.348) and hurts defensemen (0.225 vs. 0.209).
  - Rookies are slightly over-predicted (+0.007 WAR/60).
  - Their spread in notebook 17's scaled units is about normal for forwards (z SD 0.95) and wider for defensemen (1.12). Both are saved as `kind = "rookie"` rows in `dev_ranges.csv`.
  - By pre-NHL league, WHL (+0.024 WAR/60, 38 rookies), NCAA (+0.014, 35) and OHL (+0.011, 33) rookies are over-predicted, and KHL rookies are under-predicted (-0.018, 21). That's the same CHL over-prediction notebook 16 found in points.
- **The league level holds.** In 2026-27 the prior would lower the defense average by 0.0027 WAR/60 (forwards -0.0008). So the defense prior was shifted up by 0.0085 WAR/60, which brings the level back exactly.
- **The 2026-27 moves make hockey sense, with one flag.**
  - **Up:** young high-scoring forwards and recent top picks: Misa (0.033 to 0.074), Hagens, Chernyshov, Bedard (0.078 to 0.097), Celebrini (0.117 to 0.138), Jack Hughes (0.108 to 0.129), Michkov, Demidov, Lane Hutson.
  - **Down:** depth veterans with under 210 NHL minutes last season (Geertsen, LaBate, Agozzino), whose old projection borrowed the league-average forward.
  - **Red Wings:** Sandin-Pellikka +0.010 and Seider +0.004; Danielson, Finnie, Kasper essentially unchanged; Mazur -0.026.
  - **Flag, Schaefer:** his prior is low (0.016 WAR/60). His injury-shortened draft year (17 OHL games) misses the 20-game rule, so his NHLe comes from his age-16 season. His 2,024 NHL minutes keep his projection steady (0.064 to 0.061). The prospect rule should allow a shorter draft-year season or use the best season per game played.
- **Not in this step:** prospects who haven't played in the NHL aren't in our profiles (`data/bio_raw` only has NHL players). Projecting them means downloading draft classes and their profiles, a separate small step.
- **Next:** that download, a fix for short draft-year seasons, then minutes projections and the season simulator.